### Libraries

In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import pyogrio
import rasterio

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = ROOT / "data" / "raw"
NAIP_DIR = Path("/Volumes/Storage/naip")  # external drive


def check(path, script):
    """True if the file exists; otherwise say which script creates it."""
    if path.exists():
        return True
    print(f"Not downloaded yet: {path}\n  -> run python personal/download/{script}")
    return False


def sample_vector(path, layer=None, n=5):
    """Feature count, CRS, and the first n rows of a GeoPackage/GeoJSON layer (reads only n rows)."""
    info = pyogrio.read_info(path, layer=layer)
    print(f"{path.name}{f' [{layer}]' if layer else ''}: {info['features']:,} features, CRS {info['crs']}")
    return gpd.read_file(path, layer=layer, rows=n)


def sample_csv(path, n=5):
    with open(path) as f:
        n_rows = sum(1 for _ in f) - 1
    print(f"{path.name}: {n_rows:,} rows")
    return pd.read_csv(path, nrows=n)


def describe_raster(path):
    with rasterio.open(path) as src:
        print(f"{path.name}: {src.width} x {src.height} px, {src.count} bands {src.descriptions}, "
              f"{src.res[0]:.6g} {src.crs.linear_units if src.crs.is_projected else 'deg'} pixels, CRS {src.crs}")

# Data Acquisition

Sources and URLs: `data/data_sources.txt`

## 0. County boundaries

In [2]:
COUNTIES = RAW / "core_counties.geojson"

if check(COUNTIES, "download_naip.py --list-years"):
    counties = gpd.read_file(COUNTIES)
    display(counties[["NAME", "GEOID", "ALAND", "geometry"]])

,NAME,GEOID,ALAND,geometry
0,Union,34039,266160325,"POLYGON ((-74.46326 40.59918, -74.46304 40.598..."
1,Middlesex,34023,800871212,"POLYGON ((-74.63104 40.34029, -74.63099 40.340..."
2,Essex,34013,326557460,GEOMETRYCOLLECTION (LINESTRING (-74.13005 40.8...


## 1. NAIP imagery

In [3]:
NAIP_INDEX = NAIP_DIR / "naip_index.geojson"

if check(NAIP_INDEX, "download_naip.py --list-years"):
    naip_index = gpd.read_file(NAIP_INDEX)
    naip_index["downloaded"] = naip_index["path"].map(lambda p: (NAIP_DIR / p).exists())
    display(naip_index.groupby("year").agg(tiles=("id", "size"), downloaded=("downloaded", "sum"),
                                          gsd_m=("gsd", "first")))
    display(naip_index.head())

,tiles,downloaded,gsd_m
year,,,
2010,64,64,1.0
2013,64,0,1.0
2015,64,64,1.0
2017,64,0,1.0
2019,64,64,0.6
2022,64,64,0.6
2023,63,0,0.3


,id,year,datetime,gsd,path,geometry,downloaded
0,nj_m_4007424_sw_18_030_20230901_20231019,2023,2023-09-01 16:00:00+00:00,0.3,2023/nj_m_4007424_sw_18_030_20230901_20231019.tif,"POLYGON ((-74.06218 40.62426, -74.06129 40.687...",False
1,nj_m_4007424_nw_18_030_20230901_20231019,2023,2023-09-01 16:00:00+00:00,0.3,2023/nj_m_4007424_nw_18_030_20230901_20231019.tif,"POLYGON ((-74.06218 40.68676, -74.06129 40.750...",False
2,nj_m_4007414_se_18_030_20230901_20231019,2023,2023-09-01 16:00:00+00:00,0.3,2023/nj_m_4007414_se_18_030_20230901_20231019.tif,"POLYGON ((-74.24968 40.74936, -74.24897 40.812...",False
3,nj_m_4007414_ne_18_030_20230901_20231019,2023,2023-09-01 16:00:00+00:00,0.3,2023/nj_m_4007414_ne_18_030_20230901_20231019.tif,"POLYGON ((-74.24968 40.81186, -74.24896 40.875...",False
4,nj_m_4007444_sw_18_030_20230820_20231019,2023,2023-08-20 16:00:00+00:00,0.3,2023/nj_m_4007444_sw_18_030_20230820_20231019.tif,"POLYGON ((-74.56218 40.24953, -74.56177 40.312...",False


In [4]:
# One downloaded tile per year
if NAIP_INDEX.exists():
    for year, group in naip_index[naip_index["downloaded"]].groupby("year"):
        describe_raster(NAIP_DIR / group["path"].iloc[0])

nj_m_4007437_ne_18_1_20100829.tif: 5955 x 7583 px, 4 bands (None, None, None, None), 1 metre pixels, CRS EPSG:26918
nj_m_4007431_nw_18_1_20150802_20150909.tif: 5951 x 7586 px, 4 bands (None, None, None, None), 1 metre pixels, CRS EPSG:26918
nj_m_4007439_nw_18_060_20190917_20191209.tif: 9600 x 12310 px, 4 bands (None, None, None, None), 0.6 metre pixels, CRS EPSG:26918
nj_m_4007446_sw_18_060_20220720.tif: 9542 x 12229 px, 4 bands (None, None, None, None), 0.6 metre pixels, CRS EPSG:26918


## 2. Building footprints

In [5]:
OSM = RAW / "osm" / "osm_core.gpkg"

if check(OSM, "download_osm.py"):
    display(sample_vector(OSM, layer="buildings"))
    display(sample_vector(OSM, layer="industrial"))

osm_core.gpkg [buildings]: 179,420 features, CRS EPSG:4326


,osm_type,osm_id,building,name,building:levels,height,industrial,shop,amenity,operator,addr:street,addr:city,area_m2,is_large,geometry
0,relation,222683,stadium,None,None,None,None,None,None,None,None,None,40852.1,True,"MULTIPOLYGON (((-74.46571 40.51461, -74.4655 4..."
1,relation,223954,yes,South Orange Middle School,None,None,None,None,school,None,North Ridgewood Road,South Orange Village,7698.6,True,"MULTIPOLYGON (((-74.26165 40.74892, -74.26163 ..."
2,relation,1383434,school,Benjamin Franklin Elementary School,None,None,None,None,None,None,Woodbridge Avenue,Edison,5510.9,True,"MULTIPOLYGON (((-74.3804 40.5057, -74.38056 40..."
3,relation,1403696,yes,None,None,None,None,None,None,None,None,None,8298.0,True,"MULTIPOLYGON (((-74.31348 40.70655, -74.31351 ..."
4,relation,2503472,yes,School Of Engineering Building,None,None,None,None,None,None,BRETT ROAD,Piscataway,2412.5,False,"MULTIPOLYGON (((-74.46107 40.52296, -74.46105 ..."


osm_core.gpkg [industrial]: 659 features, CRS EPSG:4326


,osm_type,osm_id,landuse,industrial,name,operator,geometry
0,relation,223674,industrial,None,None,None,"MULTIPOLYGON (((-74.31826 40.87478, -74.31764 ..."
1,relation,223720,industrial,oil,Motiva Terminal,None,"MULTIPOLYGON (((-74.25399 40.54491, -74.25383 ..."
2,relation,223721,industrial,None,None,None,"MULTIPOLYGON (((-74.38969 40.54115, -74.38979 ..."
3,relation,223727,industrial,None,None,None,"MULTIPOLYGON (((-74.22274 40.61146, -74.22214 ..."
4,relation,223733,industrial,oil,Sunoco Tank Farm,None,"MULTIPOLYGON (((-74.2303 40.60917, -74.23049 4..."


In [6]:
MS_BUILDINGS = RAW / "ms_buildings" / "ms_buildings_core.gpkg"

if check(MS_BUILDINGS, "download_ms_buildings.py"):
    display(sample_vector(MS_BUILDINGS, layer="buildings"))

ms_buildings_core.gpkg [buildings]: 545,260 features, CRS EPSG:4326


,height,confidence,area_m2,is_large,geometry
0,4.574548,-1.0,151.7,False,"POLYGON ((-74.56765 40.30033, -74.5675 40.3003..."
1,-1.000000,-1.0,236.8,False,"POLYGON ((-74.56735 40.30051, -74.56744 40.300..."
2,-1.000000,-1.0,156.4,False,"POLYGON ((-74.56771 40.30066, -74.56756 40.300..."
3,-1.000000,-1.0,122.3,False,"POLYGON ((-74.56806 40.30079, -74.56818 40.300..."
4,-1.000000,-1.0,69.6,False,"POLYGON ((-74.5677 40.30082, -74.56759 40.3008..."


## 3. Parcels and land use

In [7]:
PARCELS = RAW / "parcels" / "parcels_core.gpkg"

if check(PARCELS, "download_parcels.py"):
    display(sample_vector(PARCELS, layer="parcels"))

parcels_core.gpkg [parcels]: 31,122 features, CRS EPSG:4326


,PAMS_PIN,COUNTY,MUN_NAME,PCLBLOCK,PCLLOT,PROP_LOC,PROP_CLASS,PROP_USE,BLDG_CLASS,BLDG_DESC,LAND_DESC,FAC_NAME,CALC_ACRE,YR_CONSTR,LAND_VAL,IMPRVT_VAL,NET_VALUE,SALE_PRICE,DEED_DATE,geometry
0,2001_506_6,UNION,BERKELEY HEIGHTS TWP,506,6,610 SPRINGFIELD AVE,4A,None,None,WAREHOUSE,.69 AC,None,0.69,1940.0,417300,79300,496600,1,230428,"MULTIPOLYGON (((-74.44581 40.68147, -74.44598 ..."
1,2001_1901_39,UNION,BERKELEY HEIGHTS TWP,1901,39,47 RUSSO PLACE,4B,None,None,1S S,2.210 AC,None,2.21,NaN,331500,456300,787800,1170000,101001,"MULTIPOLYGON (((-74.42651 40.68121, -74.42768 ..."
2,2001_503_1C.1_CONDO,UNION,BERKELEY HEIGHTS TWP,503,1C.1,649 SPRINGFIELD AVE,4A,670,103-A,1SCB,.770 AC,None,0.77,5876.0,200700,0,200700,700000,210504,"MULTIPOLYGON (((-74.4473 40.68172, -74.44752 4..."
3,2001_503_1C.2_CONDO,UNION,BERKELEY HEIGHTS TWP,503,1C.2,649 SPRINGFIELD AVE,4A,616,None,None,.770 AC,None,0.77,NaN,200700,181900,382600,1,161027,"MULTIPOLYGON (((-74.44682 40.68184, -74.44749 ..."
4,2001_208_1,UNION,BERKELEY HEIGHTS TWP,208,1,10 SUMMIT AVE,4B,None,None,1SCB,1.740 AC,None,1.74,NaN,348000,543000,891000,59436655,930701,"MULTIPOLYGON (((-74.44182 40.6879, -74.44309 4..."


## 4. Truck traffic (HPMS)

In [8]:
TRAFFIC = RAW / "traffic" / "hpms_core.gpkg"

if check(TRAFFIC, "download_traffic.py"):
    layers = sorted(pyogrio.list_layers(TRAFFIC)[:, 0])
    print("Layers:", layers)
    display(sample_vector(TRAFFIC, layer=layers[-1]))

Layers: ['hpms_2020', 'hpms_2022', 'hpms_2023', 'hpms_2024']
hpms_core.gpkg [hpms_2024]: 11,357 features, CRS EPSG:4326


,speed_limit,route_id,end_point,aadt_combination,begin_point,data_year,f_system,through_lanes,nhfn,sectionlength,...,aadt_single_unit,aadt_vt,aadt_single_unit_vt,nhs,facility_type,routename,aadt,routenumber,aadt_combination_vt,geometry
0,NaN,00000078__A305610,0.067,NaN,0.000,2024,1,1.0,NaN,0.067,...,NaN,None,None,NaN,4,None,NaN,NaN,None,"LINESTRING (-74.19588 40.71235, -74.19587 40.7..."
1,NaN,00000078__A305610,0.098,NaN,0.067,2024,1,2.0,NaN,0.031,...,NaN,None,None,NaN,4,None,NaN,NaN,None,"LINESTRING (-74.19566 40.71331, -74.19565 40.7..."
2,NaN,00000078__B305670,0.028,NaN,0.000,2024,1,2.0,NaN,0.028,...,NaN,None,None,NaN,4,None,NaN,NaN,None,"LINESTRING (-74.19539 40.71573, -74.19547 40.7..."
3,NaN,00000095__Y305850,0.279,NaN,0.000,2024,1,1.0,NaN,0.279,...,NaN,None,None,NaN,4,None,NaN,NaN,None,"LINESTRING (-74.15265 40.70693, -74.15284 40.7..."
4,NaN,00000095_S,50.426,NaN,0.000,2024,1,NaN,NaN,50.426,...,NaN,None,None,1.0,6,None,NaN,NaN,None,"LINESTRING (-73.95247 40.85169, -73.95588 40.8..."


## 5. Satellite NO2 (TROPOMI)

In [9]:
NO2_FILES = sorted((RAW / "no2").glob("tropomi_no2_*.tif"))

if NO2_FILES:
    for path in NO2_FILES:
        describe_raster(path)
else:
    check(RAW / "no2" / "tropomi_no2_<year>.tif", "download_no2_tropomi.py")

tropomi_no2_2019.tif: 73 x 86 px, 2 bands (None, None), 0.01 deg pixels, CRS EPSG:4326
tropomi_no2_2020.tif: 73 x 86 px, 2 bands (None, None), 0.01 deg pixels, CRS EPSG:4326
tropomi_no2_2021.tif: 73 x 86 px, 2 bands (None, None), 0.01 deg pixels, CRS EPSG:4326
tropomi_no2_2022.tif: 73 x 86 px, 2 bands (None, None), 0.01 deg pixels, CRS EPSG:4326
tropomi_no2_2023.tif: 73 x 86 px, 2 bands (None, None), 0.01 deg pixels, CRS EPSG:4326
tropomi_no2_2024.tif: 73 x 86 px, 2 bands (None, None), 0.01 deg pixels, CRS EPSG:4326
tropomi_no2_2025.tif: 73 x 86 px, 2 bands (None, None), 0.01 deg pixels, CRS EPSG:4326


## 6. Ground air quality

In [10]:
AQS_NO2 = RAW / "air_monitors" / "aqs_daily_no2.csv"
AQS_PM25 = RAW / "air_monitors" / "aqs_daily_pm25.csv"

for path in [AQS_NO2, AQS_PM25]:
    if check(path, "download_air_monitors.py"):
        display(sample_csv(path))

aqs_daily_no2.csv: 60,186 rows


,State Code,County Code,Site Num,POC,Latitude,Longitude,Sample Duration,Pollutant Standard,Date Local,Units of Measure,...,Observation Count,Observation Percent,Arithmetic Mean,1st Max Value,AQI,Method Name,Local Site Name,State Name,County Name,City Name
0,34,3,6,1,40.870436,-73.991994,1 HOUR,NO2 1-hour 2010,2010-01-01,Parts per billion,...,24,100.0,26.125000,38.0,36,INSTRUMENTAL - CHEMILUMINESCENCE,Leonia,New Jersey,Bergen,Leonia
1,34,3,6,1,40.870436,-73.991994,1 HOUR,NO2 1-hour 2010,2010-01-02,Parts per billion,...,24,100.0,4.250000,7.0,7,INSTRUMENTAL - CHEMILUMINESCENCE,Leonia,New Jersey,Bergen,Leonia
2,34,3,6,1,40.870436,-73.991994,1 HOUR,NO2 1-hour 2010,2010-01-03,Parts per billion,...,24,100.0,3.916667,5.0,5,INSTRUMENTAL - CHEMILUMINESCENCE,Leonia,New Jersey,Bergen,Leonia
3,34,3,6,1,40.870436,-73.991994,1 HOUR,NO2 1-hour 2010,2010-01-04,Parts per billion,...,24,100.0,8.916667,17.0,16,INSTRUMENTAL - CHEMILUMINESCENCE,Leonia,New Jersey,Bergen,Leonia
4,34,3,6,1,40.870436,-73.991994,1 HOUR,NO2 1-hour 2010,2010-01-05,Parts per billion,...,24,100.0,10.666667,20.0,19,INSTRUMENTAL - CHEMILUMINESCENCE,Leonia,New Jersey,Bergen,Leonia


aqs_daily_pm25.csv: 149,902 rows


,State Code,County Code,Site Num,POC,Latitude,Longitude,Sample Duration,Pollutant Standard,Date Local,Units of Measure,...,Observation Count,Observation Percent,Arithmetic Mean,1st Max Value,AQI,Method Name,Local Site Name,State Name,County Name,City Name
0,34,3,3,1,40.852256,-73.973314,24 HOUR,PM25 24-hour 2012,2010-01-02,Micrograms/cubic meter (LC),...,1,100.0,7.6,7.6,42.0,R & P Model 2025 PM2.5 Sequential w/WINS - GRA...,Fort Lee Library,New Jersey,Bergen,Fort Lee
1,34,3,3,1,40.852256,-73.973314,24 HOUR,PM25 24-hour 2012,2010-01-05,Micrograms/cubic meter (LC),...,1,100.0,8.3,8.3,46.0,R & P Model 2025 PM2.5 Sequential w/WINS - GRA...,Fort Lee Library,New Jersey,Bergen,Fort Lee
2,34,3,3,1,40.852256,-73.973314,24 HOUR,PM25 24-hour 2012,2010-01-08,Micrograms/cubic meter (LC),...,1,100.0,8.8,8.8,49.0,R & P Model 2025 PM2.5 Sequential w/WINS - GRA...,Fort Lee Library,New Jersey,Bergen,Fort Lee
3,34,3,3,1,40.852256,-73.973314,24 HOUR,PM25 24-hour 2012,2010-01-11,Micrograms/cubic meter (LC),...,1,100.0,12.3,12.3,57.0,R & P Model 2025 PM2.5 Sequential w/WINS - GRA...,Fort Lee Library,New Jersey,Bergen,Fort Lee
4,34,3,3,1,40.852256,-73.973314,24 HOUR,PM25 24-hour 2012,2010-01-14,Micrograms/cubic meter (LC),...,1,100.0,22.8,22.8,77.0,R & P Model 2025 PM2.5 Sequential w/WINS - GRA...,Fort Lee Library,New Jersey,Bergen,Fort Lee


In [11]:
FAQSD_ANNUAL = RAW / "faqsd" / "faqsd_pm25_annual.csv"
FAQSD_DAILY = sorted((RAW / "faqsd").glob("faqsd_pm25_daily_*.csv"))

if check(FAQSD_ANNUAL, "download_faqsd.py"):
    display(sample_csv(FAQSD_ANNUAL))
if FAQSD_DAILY:
    display(sample_csv(FAQSD_DAILY[-1]))

faqsd_pm25_annual.csv: 6,499 rows


,tract_fips,lon,lat,pm25_mean,n_days,year,tract_vintage
0,34013000100,-74.16911,40.78250,9.784685,365,2010,2010
1,34013000200,-74.17674,40.77143,9.760677,365,2010,2010
2,34013000300,-74.17826,40.77727,9.741827,365,2010,2010
3,34013000400,-74.18288,40.77434,9.728436,365,2010,2010
4,34013000500,-74.18364,40.76795,9.746553,365,2010,2010


faqsd_pm25_daily_2022.csv: 190,895 rows


,date,tract_fips,lon,lat,pm25,pm25_stderr
0,2022/01/01,34013000100,-74.16913,40.78248,8.660,3.8249
1,2022/01/01,34013000200,-74.17677,40.77143,8.559,3.5318
2,2022/01/01,34013000300,-74.17827,40.77727,8.599,3.6025
3,2022/01/01,34013000400,-74.18289,40.77433,8.599,3.7029
4,2022/01/01,34013000500,-74.18366,40.76799,8.678,3.7596


In [12]:
PM25_SAT_FILES = sorted((RAW / "pm25_satellite").glob("acag_pm25_*.tif"))

if PM25_SAT_FILES:
    for path in PM25_SAT_FILES:
        describe_raster(path)
else:
    check(RAW / "pm25_satellite" / "acag_pm25_<year>.tif", "download_pm25_satellite.py")

acag_pm25_2010.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2011.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2012.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2013.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2014.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2015.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2016.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2017.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2018.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2019.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2020.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2021.tif: 73 x 86 px, 1 bands (None,), 0.01 deg pixels, CRS EPSG:4326
acag_pm25_2022.tif: 73 x 86 px, 1 bands 

## 7. Asthma ER visits (manual, NJSHAD)
Asthma ED visit counts by county of residence, 2016–2024. NJSHAD offers no rates or geography smaller than county.

In [13]:
ASTHMA = RAW / "health" / "asthma_dataset.xlsx"

if ASTHMA.exists():
    # NJSHAD export: county name rows, then one row per county-year; "**" = suppressed small count
    asthma = pd.read_excel(ASTHMA)
    asthma.columns = ["group", "county", "year", "ed_visits", "footnote"]
    asthma = asthma[asthma["year"].notna() & (asthma["county"] != "Unknown Residence")]
    asthma["year"] = asthma["year"].astype(int)
    asthma["ed_visits"] = pd.to_numeric(asthma["ed_visits"].astype(str).str.replace(",", ""), errors="coerce")
    asthma = asthma[["county", "year", "ed_visits"]].reset_index(drop=True)
    print(f"{ASTHMA.name}: {asthma['county'].nunique()} counties, {asthma['year'].min()}-{asthma['year'].max()}")
    display(asthma.pivot(index="county", columns="year", values="ed_visits").loc[["Essex", "Middlesex", "Union"]])
else:
    print("No asthma export yet; save the NJSHAD export as data/raw/health/asthma_dataset.xlsx")

asthma_dataset.xlsx: 21 counties, 2016-2024


year,2016,2017,2018,2019,2020,2021,2022,2023,2024
county,,,,,,,,,
Essex,9091,8464,8159,7860,4373,5379,5764,5938,5744
Middlesex,3233,3101,3087,2926,1510,1937,2372,2415,2333
Union,3208,3179,3049,2775,1503,1893,2393,2517,2289


## 8. Demographics (Census ACS)

In [14]:
TRACTS = RAW / "census" / "tracts_acs_2024.gpkg"

if check(TRACTS, "download_census.py"):
    display(sample_vector(TRACTS, layer="tracts"))

tracts_acs_2024.gpkg [tracts]: 523 features, CRS EPSG:4269


,GEOID,COUNTYFP,ALAND,NAME,total_pop,nh_white,nh_black,nh_asian,hispanic,median_hh_income,pov_universe,pov_ratio_2x_plus,pct_minority,pct_low_income,geometry
0,34039032004,039,232569,Census Tract 320.04; Union County; New Jersey,2584,204,130,437,1725,65951.0,2583,1715,0.921053,0.336043,"POLYGON ((-74.24347 40.66282, -74.24005 40.663..."
1,34039032401,039,814882,Census Tract 324.01; Union County; New Jersey,4268,1240,849,152,1525,111250.0,4187,3117,0.709466,0.255553,"POLYGON ((-74.23766 40.6898, -74.23762 40.6898..."
2,34039032102,039,765935,Census Tract 321.02; Union County; New Jersey,3333,1812,1,226,1249,140682.0,3333,3022,0.456346,0.093309,"POLYGON ((-74.25407 40.67025, -74.25315 40.670..."
3,34039039501,039,590924,Census Tract 395.01; Union County; New Jersey,3364,8,1081,0,2257,87500.0,3302,1982,0.997622,0.399758,"POLYGON ((-74.44359 40.60069, -74.4429 40.6011..."
4,34039032101,039,840475,Census Tract 321.01; Union County; New Jersey,3605,1417,159,0,1876,131607.0,3605,2718,0.606935,0.246047,"POLYGON ((-74.24909 40.66755, -74.24802 40.667..."


## 9. Environmental justice layers

In [15]:
OBC = RAW / "ej" / "overburdened_communities.gpkg"
EJSCREEN = RAW / "ej" / "ejscreen_2024_tracts_core.csv"

if check(OBC, "download_ej_layers.py"):
    display(sample_vector(OBC, layer="obc"))
if check(EJSCREEN, "download_ej_layers.py"):
    display(sample_csv(EJSCREEN))

overburdened_communities.gpkg [obc]: 3,752 features, CRS EPSG:4326


,OBJECTID,GEOID,POVUNIVERSE,POPUNDER2XPOV,LOW_INCOME_PCT,TOTALPOP,NONHISPWHITE,TOTALMINORITY,MINORITY_PCT,TOTHH,...,GEOID_COUNTYSUBDIVISION,COUNTY,NAME,MUN_CODE,SPLIT_BG,SPLIT_MUN,GLOBALID,Shape__Area,Shape__Length,geometry
0,1,340030311003,2202,239,10.853769,2202,1238,964,43.778383,974,...,3400342090,BERGEN,Lyndhurst Township,0232,Not a split block group,Not Applicable,2f8c3d17-a4fb-48af-80ab-54ddc61b85f2,4.042237e+06,8798.685129,"MULTIPOLYGON (((-74.11634 40.81175, -74.11555 ..."
1,2,340030314001,678,168,24.778761,678,381,297,43.805310,287,...,3400342090,BERGEN,Lyndhurst Township,0232,Not a split block group,Not Applicable,e89dd5ac-80d4-4d3e-96e5-0ddc5206bdc7,2.787376e+06,6670.532314,"MULTIPOLYGON (((-74.11385 40.80579, -74.11444 ..."
2,3,340030314004,975,68,6.974359,975,429,546,56.000000,420,...,3400342090,BERGEN,Lyndhurst Township,0232,Not a split block group,Not Applicable,8c9e9b60-b4ef-408f-bedf-28ba36701b7d,1.781400e+06,6270.401219,"MULTIPOLYGON (((-74.12414 40.80102, -74.1243 4..."
3,4,340030331002,798,90,11.278195,798,410,388,48.621554,287,...,3400344880,BERGEN,Maywood Borough,0234,Not a split block group,Not Applicable,1932d39f-89df-4272-bec0-17f27fa0ebb2,2.334516e+06,6420.378033,"MULTIPOLYGON (((-74.05442 40.90634, -74.05449 ..."
4,5,340030331004,1629,104,6.384285,1629,916,713,43.769184,497,...,3400344880,BERGEN,Maywood Borough,0234,Not a split block group,Not Applicable,51a29c69-dce6-43f2-8b82-53428313d12f,4.291356e+06,12533.233900,"MULTIPOLYGON (((-74.05705 40.91302, -74.05713 ..."


ejscreen_2024_tracts_core.csv: 523 rows


,OID_,ID,STATE_NAME,ST_ABBREV,CNTY_NAME,REGION,ACSTOTPOP,ACSIPOVBAS,ACSEDUCBAS,ACSTOTHH,...,T_D2_DWATER,T_D5_DWATER,AREALAND,AREAWATER,NPL_CNT,TSDF_CNT,EXCEED_COUNT_80,EXCEED_COUNT_80_SUP,Shape_Length,Shape_Area
0,47539,34013000100,NEW JERSEY,NJ,Essex County,2,6576.0,6531.0,4790.0,2655.0,...,0 %ile,0 %ile,938930.0,6653.0,0.0,0.0,10,5.0,0.043578,0.000101
1,47540,34013000200,NEW JERSEY,NJ,Essex County,2,3444.0,3430.0,2456.0,1291.0,...,0 %ile,0 %ile,763973.0,22301.0,0.0,0.0,11,11.0,0.040686,0.000084
2,47541,34013000300,NEW JERSEY,NJ,Essex County,2,4423.0,4423.0,2912.0,1987.0,...,93 %ile,94 %ile,327802.0,0.0,0.0,0.0,12,12.0,0.045388,0.000035
3,47542,34013000400,NEW JERSEY,NJ,Essex County,2,2383.0,2375.0,1634.0,912.0,...,0 %ile,0 %ile,178899.0,0.0,0.0,0.0,11,11.0,0.021911,0.000019
4,47543,34013000500,NEW JERSEY,NJ,Essex County,2,2152.0,2152.0,1305.0,666.0,...,0 %ile,0 %ile,207629.0,0.0,0.0,0.0,11,11.0,0.021500,0.000022


## 10. Planned warehouses (manual, optional)

In [16]:
PLANNED = RAW / "planned_warehouses.csv"

if PLANNED.exists():
    display(sample_csv(PLANNED))
else:
    print("No planned_warehouses.csv yet (optional)")

No planned_warehouses.csv yet (optional)
